In [10]:
import numpy as np
import matplotlib.pyplot as plt
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI

def create_simple_causal_graphs():
    """
    Create and plot simple causal graph structures.
    """
    def lin(x): return x

    links = {0: [],   # X
             1: [],                                             # Y
             2: [((0, 0), 0.3, lin), ((1, 0), 0.4, lin)],                        # Z                     
            }
    
    var_names = [r'$X^{%d}$' % j for j in range(1, len(links)+1)]
    
    # Show ground truth causal graph
    tp.plot_graph(
        graph = PCMCI.get_graph_from_dict(links),
        var_names=var_names,
        )
    
    # Create plots
    for name, graph in graphs.items():
        plt.figure(figsize=(6, 4))
        # Create a value matrix of the same shape filled with 1s
        val_matrix = np.ones_like(graph, dtype=float)
        tp.plot_graph(val_matrix, graph, var_names=['X', 'Y', 'Z'])
        plt.title(f'Causal Graph: {name}')
        plt.tight_layout()
        #plt.savefig(f'{name}_causal_graph.png')
        plt.show()
        #plt.close()

def create_time_series_graph(auto_coeff=0.8, cross_coeff=0.5):
    """
    Create a time series graph plot with color-coded auto and cross links.
    """
    # Define the links structure
    links = {
        0: [((0, -1), auto_coeff, lambda x: x), 
            ((2, -1), cross_coeff, lambda x: x)],
        1: [((1, -1), auto_coeff, lambda x: x), 
            ((0, -1), cross_coeff, lambda x: x)],
        2: [((2, -1), auto_coeff, lambda x: x)],       
        3: [((3, -1), auto_coeff, lambda x: x),
            ((0, -1), cross_coeff, lambda x: x), 
            ((1, -1), cross_coeff, lambda x: x), 
            ((2, -2), cross_coeff, lambda x: x)]
    }
    
    # Convert links to graph
    n_vars = max(links.keys()) + 1
    max_lag = max(max(abs(lag) for (_, lag), _, _ in var_links) for var_links in links.values()) + 1
    graph = np.zeros((n_vars, n_vars, max_lag), dtype=int)
    val_matrix = np.zeros_like(graph, dtype=float)
    
    for var, var_links in links.items():
        for (parent_var, parent_lag), coeff, _ in var_links:
            if parent_var != var:
                # Cross-link
                graph[var, parent_var, abs(parent_lag)] = 1
                val_matrix[var, parent_var, abs(parent_lag)] = coeff
            else:
                # Auto-link
                graph[var, parent_var, abs(parent_lag)] = 2
                val_matrix[var, parent_var, abs(parent_lag)] = coeff
    
    # Var names
    var_names = [f'X{i+1}' for i in range(n_vars)]
    
    # Create plot
    plt.figure(figsize=(10, 6))
    tp.plot_time_series_graph(
        graph=graph,
        var_names=var_names,
        link_colorbar_label='Link Strength'
    )
    plt.title('Time Series Causal Graph')
    plt.tight_layout()
    #plt.savefig('time_series_causal_graph.png')
    #plt.close()

def create_dag_plot(auto_coeff=0.8, cross_coeff=0.5):
    """
    Create a standard DAG plot for the time series structure.
    """
    # Define the links structure
    links = {
        0: [((0, -1), auto_coeff, lambda x: x), 
            ((2, -1), cross_coeff, lambda x: x)],
        1: [((1, -1), auto_coeff, lambda x: x), 
            ((0, -1), cross_coeff, lambda x: x)],
        2: [((2, -1), auto_coeff, lambda x: x)],       
        3: [((3, -1), auto_coeff, lambda x: x),
            ((0, -1), cross_coeff, lambda x: x), 
            ((1, -1), cross_coeff, lambda x: x), 
            ((2, -2), cross_coeff, lambda x: x)]
    }
    
    # Convert links to graph
    n_vars = max(links.keys()) + 1
    graph = np.zeros((n_vars, n_vars), dtype=int)
    val_matrix = np.zeros_like(graph, dtype=float)
    
    for var, var_links in links.items():
        for (parent_var, _), coeff, _ in var_links:
            if parent_var != var:
                graph[var, parent_var] = 1
                val_matrix[var, parent_var] = coeff
    
    # Var names
    var_names = [f'X{i+1}' for i in range(n_vars)]
    
    # Create plot
    plt.figure(figsize=(8, 6))
    tp.plot_graph(
        graph=graph, 
        var_names=var_names
    )
    plt.title('DAG Causal Graph')
    plt.tight_layout()
    #plt.savefig('dag_causal_graph.png')
    #plt.close()

def main():
    # Generate all plots
    #create_simple_causal_graphs()
    create_time_series_graph()
    create_dag_plot()
    print("All plots have been generated successfully!")



In [44]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_causal_structures():
    """
    Generate and plot three causal graph structures with improved layout:
    1. Confounder (X->Z<-Y)
    2. Mediator (X->Z->Y)
    3. Collider (X<-Z->Y)
    Save each plot as a separate PNG file
    """
    # Set up common parameters
    plt.rcParams['figure.figsize'] = (5, 4)
    plt.rcParams['figure.autolayout'] = True
    plt.rcParams['figure.dpi'] = 300

    # 1. Confounder Structure (X->Z<-Y)
    plt.figure()
    G1 = nx.DiGraph()
    G1.add_edges_from([('X', 'Z'), ('Y', 'Z')])
    pos1 = {'X': (0, 0), 'Y': (2, 0), 'Z': (1, 1)}
    nx.draw(G1, pos1, with_labels=True, node_color='lightblue', 
            node_size=2000, arrowsize=20, font_size=10, font_weight='bold')
    plt.title('Collider')
    plt.axis('equal')
    plt.savefig('collider.png', bbox_inches='tight', pad_inches=0.1)
    plt.close()
    
    # 2. Mediator Structure (X->Z->Y)
    plt.figure()
    G2 = nx.DiGraph()
    G2.add_edges_from([('X', 'Z'), ('Z', 'Y')])
    pos2 = {'X': (0, 0), 'Y': (2, 0), 'Z': (1, 1)}
    nx.draw(G2, pos2, with_labels=True, node_color='lightblue', 
            node_size=2000, arrowsize=20, font_size=10, font_weight='bold')
    plt.title('Mediator')
    plt.axis('equal')
    plt.savefig('mediator.png', bbox_inches='tight', pad_inches=0.1)
    plt.close()
    
    # 3. Collider Structure (X<-Z->Y)
    plt.figure()
    G3 = nx.DiGraph()
    G3.add_edges_from([('Z', 'X'), ('Z', 'Y')])
    pos3 = {'X': (0, 0), 'Y': (2, 0), 'Z': (1, 1)}
    nx.draw(G3, pos3, with_labels=True, node_color='lightblue', 
            node_size=2000, arrowsize=20, font_size=10, font_weight='bold')
    plt.title('Confounder')
    plt.axis('equal')
    plt.savefig('confounder.png', bbox_inches='tight', pad_inches=0.1)
    plt.close()

# Call the function to generate and save the plots
plot_causal_structures()

print("Plots have been saved as confounder.png, mediator.png, and collider.png")

C:\Users\21ws_fuller\AppData\Local\Temp\ipykernel_27428\2954162027.py:26: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.savefig('collider.png', bbox_inches='tight', pad_inches=0.1)
C:\Users\21ws_fuller\AppData\Local\Temp\ipykernel_27428\2954162027.py:38: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.savefig('mediator.png', bbox_inches='tight', pad_inches=0.1)


Plots have been saved as confounder.png, mediator.png, and collider.png


C:\Users\21ws_fuller\AppData\Local\Temp\ipykernel_27428\2954162027.py:50: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.savefig('confounder.png', bbox_inches='tight', pad_inches=0.1)


In [1]:
import numpy as np

In [8]:
type(np.repeat([10,50,100,200,500],5).tolist()[0])

int

In [7]:
np.repeat(np.linspace(0.01,.99,20),5).tolist()


[0.01,
 0.01,
 0.01,
 0.01,
 0.01,
 0.06157894736842105,
 0.06157894736842105,
 0.06157894736842105,
 0.06157894736842105,
 0.06157894736842105,
 0.1131578947368421,
 0.1131578947368421,
 0.1131578947368421,
 0.1131578947368421,
 0.1131578947368421,
 0.16473684210526315,
 0.16473684210526315,
 0.16473684210526315,
 0.16473684210526315,
 0.16473684210526315,
 0.2163157894736842,
 0.2163157894736842,
 0.2163157894736842,
 0.2163157894736842,
 0.2163157894736842,
 0.26789473684210524,
 0.26789473684210524,
 0.26789473684210524,
 0.26789473684210524,
 0.26789473684210524,
 0.3194736842105263,
 0.3194736842105263,
 0.3194736842105263,
 0.3194736842105263,
 0.3194736842105263,
 0.37105263157894736,
 0.37105263157894736,
 0.37105263157894736,
 0.37105263157894736,
 0.37105263157894736,
 0.4226315789473684,
 0.4226315789473684,
 0.4226315789473684,
 0.4226315789473684,
 0.4226315789473684,
 0.47421052631578947,
 0.47421052631578947,
 0.47421052631578947,
 0.47421052631578947,
 0.47421052631578